# Chương 4. Dự đoán giá vàng XAU/USD bằng RNN, LSTM và GRU

Notebook dự đoán **giá Close của phiên kế tiếp** từ 60 phiên liên tiếp trước đó. Đây là bài toán hồi quy chuỗi thời gian: đầu vào là cửa sổ giá OHLC theo thứ tự thời gian; đầu ra là một giá trị Close. Ba mạng hồi quy được so sánh trên cùng dữ liệu, cách chia tập và quy trình đánh giá.

**Nguồn dữ liệu:** [XAU/USD Gold Price Historical Data trên Kaggle](https://www.kaggle.com/datasets/novandraanugrah/xauusd-gold-price-historical-data-2004-2024). Tải và giải nén các CSV vào thư mục `data/` cạnh notebook. Notebook ưu tiên tệp ngày (`1d`, `daily`). Dữ liệu có thể được cập nhật sau thời điểm viết notebook, vì vậy số dòng và kết quả được tính khi chạy.

Chạy lần lượt từ trên xuống dưới. Cài thư viện bằng `pip install -r requirements_gold.txt` trong cùng môi trường với Jupyter.

---
## 01. Thư viện và cấu hình

Đổi `DATA_PATH` nếu muốn dùng một CSV cụ thể. Khi để `None`, notebook tự tìm CSV ngày trong `data/`. Seed cố định giúp giảm biến động giữa các lần chạy; CPU/GPU vẫn có thể cho sai khác nhỏ.

In [ ]:
from pathlib import Path
import copy
import random
import re

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler
from torch import nn
from torch.utils.data import DataLoader, Dataset

DATA_PATH = None                  # Ví dụ: Path("data/XAU_1d_data.csv")
DATA_DIR = Path("data")
ARTIFACT_DIR = Path("artifacts")
SEQUENCE_LENGTH = 60
BATCH_SIZE = 64
HIDDEN_SIZE = 64
NUM_LAYERS = 2
DROPOUT = 0.2
LEARNING_RATE = 0.001
MAX_EPOCHS = 50
PATIENCE = 8
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Thiết bị: {device} | seed: {SEED} | cửa sổ: {SEQUENCE_LENGTH}")

---
## 02. Đọc dữ liệu

Chỉ chọn CSV có tên thể hiện khung thời gian ngày. Hiển thị thông tin đủ để kiểm tra dữ liệu mà không in toàn bộ bảng.

In [ ]:
def find_daily_csv(data_path, data_dir):
    if data_path is not None:
        path = Path(data_path)
        if not path.is_file():
            raise FileNotFoundError(f"Không tìm thấy DATA_PATH: {path}")
        return path
    files = list(data_dir.rglob("*.csv")) if data_dir.exists() else []
    daily = [p for p in files if re.search(r"(^|[_\W])(1d|daily|day)([_\W]|$)", p.stem.lower())]
    if not daily:
        raise FileNotFoundError(
            "Chưa có CSV ngày. Giải nén Kaggle vào data/ hoặc đặt DATA_PATH tới CSV ngày."
        )
    return sorted(daily, key=lambda p: ("1d" not in p.stem.lower(), str(p).lower()))[0]


csv_path = find_daily_csv(DATA_PATH, DATA_DIR)
raw = pd.read_csv(csv_path, sep=None, engine="python")
print(f"Tệp: {csv_path} | kích thước: {raw.shape} | cột: {list(raw.columns)}")
display(raw.head())
display(pd.DataFrame({"kiểu dữ liệu": raw.dtypes.astype(str), "thiếu": raw.isna().sum()}))
print(f"Dòng trùng hoàn toàn: {raw.duplicated().sum()}")

---
## 03. Chuẩn hóa cột và làm sạch

Chuẩn hóa tên cột viết hoa/thường; chuyển ngày và giá sang kiểu phù hợp; bỏ dòng thiếu OHLC và giữ bản ghi cuối nếu trùng ngày. Không điền giá tương lai vào quá khứ.

In [ ]:
def canonical_name(name):
    key = re.sub(r"[^a-z0-9]", "", str(name).lower())
    aliases = {
        "date": "Date", "datetime": "Date", "timestamp": "Date", "time": "Date",
        "open": "Open", "high": "High", "low": "Low", "close": "Close",
        "volume": "Volume", "tickvol": "Volume", "tickvolume": "Volume",
    }
    return aliases.get(key, str(name).strip())


df = raw.copy()
df.columns = [canonical_name(col) for col in df.columns]
if df.columns.duplicated().any():
    df = df.loc[:, ~df.columns.duplicated()].copy()
required = ["Date", "Open", "High", "Low", "Close"]
missing_cols = [col for col in required if col not in df.columns]
if missing_cols:
    raise ValueError(f"CSV ngày thiếu các cột {missing_cols}; cột hiện có: {list(df.columns)}")

n_before = len(df)
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
for col in ["Open", "High", "Low", "Close"]:
    df[col] = pd.to_numeric(df[col].astype(str).str.replace(",", "", regex=False), errors="coerce")
df = df.dropna(subset=required).sort_values("Date")
duplicates_by_date = int(df.duplicated(subset="Date").sum())
df = df.drop_duplicates(subset="Date", keep="last").reset_index(drop=True)
df = df[(df[["Open", "High", "Low", "Close"]] > 0).all(axis=1)].reset_index(drop=True)
if len(df) <= SEQUENCE_LENGTH + 20:
    raise ValueError("Dữ liệu hợp lệ quá ngắn để chia tập và tạo cửa sổ 60 phiên.")

display(pd.DataFrame([{
    "dòng gốc": n_before, "dòng hợp lệ": len(df), "ngày trùng": duplicates_by_date,
    "từ ngày": df["Date"].min().date(), "đến ngày": df["Date"].max().date(),
}]))

---
## 04. Khám phá dữ liệu (EDA)

Xem thống kê, diễn biến Close, OHLC và phân bố Close. Xu hướng trên đồ thị chỉ là mô tả dữ liệu; đánh giá khả năng dự đoán dùng tập test ở các mục sau.

In [ ]:
display(df[["Open", "High", "Low", "Close"]].describe().T.round(2))

fig, axes = plt.subplots(3, 1, figsize=(12, 10))
axes[0].plot(df["Date"], df["Close"], linewidth=1)
axes[0].set(title="Giá Close theo thời gian", ylabel="USD/oz")
for col in ["Open", "High", "Low", "Close"]:
    axes[1].plot(df["Date"], df[col], label=col, linewidth=0.8)
axes[1].set(title="Các giá OHLC", ylabel="USD/oz")
axes[1].legend(ncol=4)
axes[2].hist(df["Close"], bins=40, edgecolor="white")
axes[2].set(title="Phân bố giá Close", xlabel="USD/oz", ylabel="Số phiên")
fig.tight_layout()
plt.show()

**Nhận xét EDA.** Quan sát đồ thị trên để xác định các giai đoạn tăng/giảm và vùng giá biến động mạnh. Phân bố Close theo toàn bộ thời gian có thể lệch vì mức giá thay đổi theo giai đoạn; điều đó không tự chứng minh mô hình dự đoán tốt.

---
## 05. Chia train / validation / test theo thời gian

Chia 70% / 15% / 15% theo **ngày đích**. Không xáo trộn thứ tự trước khi chia. Mẫu đầu của validation/test được phép dùng 60 ngày *trước đó* làm ngữ cảnh, vì đây là dữ liệu đã biết tại thời điểm dự đoán.

In [ ]:
n_rows = len(df)
train_end = int(n_rows * 0.70)
val_end = int(n_rows * 0.85)
if train_end <= SEQUENCE_LENGTH or val_end <= train_end or val_end >= n_rows:
    raise ValueError("Không đủ dữ liệu cho phép chia 70/15/15 và cửa sổ đã chọn.")
split_info = pd.DataFrame({
    "tập": ["Train", "Validation", "Test"],
    "số ngày đích": [train_end - SEQUENCE_LENGTH, val_end - train_end, n_rows - val_end],
    "bắt đầu": [df.loc[SEQUENCE_LENGTH, "Date"], df.loc[train_end, "Date"], df.loc[val_end, "Date"]],
    "kết thúc": [df.loc[train_end - 1, "Date"], df.loc[val_end - 1, "Date"], df.loc[n_rows - 1, "Date"]],
})
display(split_info)

---
## 06. Chuẩn hóa trên train

Feature gồm `Open`, `High`, `Low`, `Close`; target là `Close`. Hai `MinMaxScaler` chỉ `fit` trên 70% dữ liệu train. Validation/test chỉ dùng `transform`, tránh rò rỉ thống kê tương lai.

In [ ]:
FEATURES = ["Open", "High", "Low", "Close"]
feature_scaler = MinMaxScaler()
target_scaler = MinMaxScaler()
feature_scaler.fit(df.loc[:train_end - 1, FEATURES])
target_scaler.fit(df.loc[:train_end - 1, ["Close"]])

features_scaled = feature_scaler.transform(df[FEATURES]).astype(np.float32)
target_scaled = target_scaler.transform(df[["Close"]]).astype(np.float32)
sequence_data = np.column_stack([features_scaled, target_scaled])
display(pd.DataFrame({"feature": FEATURES, "min train": feature_scaler.data_min_, "max train": feature_scaler.data_max_}).round(2))

---
## 07. Tạo cửa sổ trượt

`create_sequences(data, sequence_length)` tạo 60 giá trị liên tiếp làm `X` và Close phiên thứ 61 làm `y`. Mỗi cửa sổ chỉ chứa dữ liệu **trước** ngày đích; phân tập theo chỉ số ngày đích.

In [ ]:
def create_sequences(data, sequence_length):
    X, y = [], []
    for target_day in range(sequence_length, len(data)):
        X.append(data[target_day - sequence_length:target_day, :-1])
        y.append(data[target_day, -1])
    return np.stack(X).astype(np.float32), np.asarray(y, dtype=np.float32).reshape(-1, 1)


X_all, y_all = create_sequences(sequence_data, SEQUENCE_LENGTH)
target_days = np.arange(SEQUENCE_LENGTH, n_rows)
train_mask = target_days < train_end
val_mask = (target_days >= train_end) & (target_days < val_end)
test_mask = target_days >= val_end
X_train, y_train = X_all[train_mask], y_all[train_mask]
X_val, y_val = X_all[val_mask], y_all[val_mask]
X_test, y_test = X_all[test_mask], y_all[test_mask]
test_dates = df.loc[target_days[test_mask], "Date"].reset_index(drop=True)
display(pd.DataFrame({
    "tập": ["Train", "Validation", "Test"],
    "shape X": [str(x.shape) for x in [X_train, X_val, X_test]],
    "shape y": [str(y.shape) for y in [y_train, y_val, y_test]],
}))

---
## 08. PyTorch Dataset và DataLoader

Batch có dạng `[batch_size, sequence_length, input_size]`. Chỉ train được shuffle giữa các cửa sổ đã tạo; validation và test giữ nguyên thứ tự.

In [ ]:
class TimeSeriesDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        return self.X[index], self.y[index]


train_loader = DataLoader(TimeSeriesDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TimeSeriesDataset(X_val, y_val), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(TimeSeriesDataset(X_test, y_test), batch_size=BATCH_SIZE, shuffle=False)
print(f"Số batch: train={len(train_loader)}, validation={len(val_loader)}, test={len(test_loader)}")

---
## 09. SimpleRNN

`nn.RNN` là mạng hồi quy cơ bản. Lấy trạng thái tại bước thời gian cuối và đưa qua lớp tuyến tính để dự đoán một giá Close.

In [ ]:
class SimpleRNN(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2, output_size=1):
        super().__init__()
        self.recurrent = nn.RNN(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        self.output = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # x: [batch_size, sequence_length, input_size]
        states, _ = self.recurrent(x)
        return self.output(states[:, -1, :])

---
## 10. LSTM

`nn.LSTM` dùng trạng thái ô nhớ và các cổng để giữ hoặc bỏ thông tin theo thời gian. Kích thước hidden, số tầng và dropout giống SimpleRNN.

In [ ]:
class LSTM(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2, output_size=1):
        super().__init__()
        self.recurrent = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        self.output = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # x: [batch_size, sequence_length, input_size]
        states, _ = self.recurrent(x)
        return self.output(states[:, -1, :])

---
## 11. GRU

`nn.GRU` cũng dùng cổng để xử lý phụ thuộc thời gian, với cấu trúc gọn hơn LSTM. Giữ các tham số huấn luyện tương đương để so sánh.

In [ ]:
class GRU(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2, output_size=1):
        super().__init__()
        self.recurrent = nn.GRU(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        self.output = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # x: [batch_size, sequence_length, input_size]
        states, _ = self.recurrent(x)
        return self.output(states[:, -1, :])

---
## 12. Hàm huấn luyện chung

Dùng MSELoss và Adam (`learning rate = 0.001`). Lưu trọng số ở epoch có validation loss thấp nhất và dừng sớm sau 8 epoch không cải thiện. Không in log mỗi epoch để giữ đầu ra gọn.

In [ ]:
def train_model(model, train_loader, val_loader, max_epochs=MAX_EPOCHS, patience=PATIENCE):
    model = model.to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    history = {"train": [], "val": []}
    best_loss = float("inf")
    best_state = None
    waiting = 0

    for _ in range(max_epochs):
        model.train()
        train_sum = 0.0
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(batch_X), batch_y)
            loss.backward()
            optimizer.step()
            train_sum += loss.item() * len(batch_X)
        train_loss = train_sum / len(train_loader.dataset)

        model.eval()
        val_sum = 0.0
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                val_sum += criterion(model(batch_X), batch_y).item() * len(batch_X)
        val_loss = val_sum / len(val_loader.dataset)
        history["train"].append(train_loss)
        history["val"].append(val_loss)

        if val_loss < best_loss:
            best_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            waiting = 0
        else:
            waiting += 1
            if waiting >= patience:
                break

    model.load_state_dict(best_state)
    return model, history

---
## 13. Huấn luyện ba mô hình

Cả ba dùng cùng cửa sổ, batch size, số tầng, hidden size, dropout, optimizer, learning rate và tối đa 50 epoch. Bảng tóm tắt chỉ hiển thị epoch tốt nhất và validation loss.

In [ ]:
model_classes = {"SimpleRNN": SimpleRNN, "LSTM": LSTM, "GRU": GRU}
models, histories, training_rows = {}, {}, []
for name, model_class in model_classes.items():
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    model = model_class(len(FEATURES), HIDDEN_SIZE, NUM_LAYERS, DROPOUT)
    models[name], histories[name] = train_model(model, train_loader, val_loader)
    history = histories[name]
    best_epoch = int(np.argmin(history["val"])) + 1
    training_rows.append({
        "Model": name, "Epoch tốt nhất": best_epoch, "Epoch đã chạy": len(history["val"]),
        "Validation MSE chuẩn hóa": min(history["val"]),
    })
display(pd.DataFrame(training_rows).round(6))

---
## 14. Biểu đồ loss

Mỗi ô thể hiện train và validation loss của một mô hình. Khoảng cách lớn và kéo dài giữa hai đường là một dấu hiệu cần kiểm tra overfitting.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (name, history) in zip(axes, histories.items()):
    epochs = np.arange(1, len(history["train"]) + 1)
    ax.plot(epochs, history["train"], label="Train")
    ax.plot(epochs, history["val"], label="Validation")
    ax.set(title=name, xlabel="Epoch", ylabel="MSE trên dữ liệu chuẩn hóa")
    ax.legend()
fig.tight_layout()
plt.show()

---
## 15. Dự đoán test và hoàn nguyên đơn vị

Tập test không tham gia huấn luyện hoặc chọn epoch. Chuyển dự đoán và giá thật về USD/oz trước khi tính metric.

In [ ]:
def predict_scaled(model, loader):
    model.eval()
    chunks = []
    with torch.no_grad():
        for batch_X, _ in loader:
            chunks.append(model(batch_X.to(device)).cpu().numpy())
    return np.vstack(chunks)


y_actual = target_scaler.inverse_transform(y_test).ravel()
predictions = {}
for name, model in models.items():
    pred_scaled = predict_scaled(model, test_loader)
    predictions[name] = target_scaler.inverse_transform(pred_scaled).ravel()

---
## 16. Đánh giá và xếp hạng

MAE, MSE và RMSE có đơn vị giá gốc; R² không có đơn vị. MAPE được tính trên các giá Close khác 0. Xếp hạng theo RMSE thấp nhất.

In [ ]:
metric_rows = []
for name, y_pred in predictions.items():
    mse = mean_squared_error(y_actual, y_pred)
    metric_rows.append({
        "Model": name,
        "MAE": mean_absolute_error(y_actual, y_pred),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "R2": r2_score(y_actual, y_pred),
        "MAPE (%)": np.mean(np.abs((y_actual - y_pred) / y_actual)) * 100,
    })
results = pd.DataFrame(metric_rows).sort_values("RMSE").reset_index(drop=True)
best_name = str(results.loc[0, "Model"])
best_model = models[best_name]
display(results.round(3))

---
## 17. Actual vs Predicted

Biểu đồ riêng cho từng mô hình trên toàn bộ test; biểu đồ cuối phóng to tối đa 150 phiên cuối để dễ nhìn độ bám giá.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
for ax, (name, y_pred) in zip(axes, predictions.items()):
    ax.plot(test_dates, y_actual, label="Actual", linewidth=1)
    ax.plot(test_dates, y_pred, label=name, linewidth=1)
    ax.set(title=name, ylabel="USD/oz")
    ax.legend()
fig.tight_layout()
plt.show()

last = min(150, len(y_actual))
plt.figure(figsize=(13, 4))
plt.plot(test_dates.iloc[-last:], y_actual[-last:], label="Actual", linewidth=2)
for name, y_pred in predictions.items():
    plt.plot(test_dates.iloc[-last:], y_pred[-last:], label=name, linewidth=1)
plt.title(f"So sánh trên {last} phiên test cuối")
plt.ylabel("USD/oz")
plt.legend()
plt.tight_layout()
plt.show()

---
## 18. Nhận xét kết quả

SimpleRNN đơn giản, nhưng trạng thái qua nhiều bước dễ mất thông tin. LSTM dùng ô nhớ và các cổng để học phụ thuộc dài hạn; GRU có cơ chế cổng gọn hơn. Mô hình tốt nhất **trên tập dữ liệu này** được xác định từ bảng test, không giả định trước. So sánh train/validation loss để tìm dấu hiệu overfitting; một khoảng cách lớn là tín hiệu, chưa đủ để kết luận chắc chắn. Giá vàng còn chịu tác động của tin tức, lãi suất và biến động thị trường mà chuỗi OHLC quá khứ không nắm bắt hết.

In [ ]:
best_history = histories[best_name]
best_epoch_index = int(np.argmin(best_history["val"]))
train_at_best = best_history["train"][best_epoch_index]
val_at_best = best_history["val"][best_epoch_index]
gap = val_at_best / max(train_at_best, 1e-12)
display(Markdown(
    f"**Kết quả trên test:** {best_name} có RMSE thấp nhất "
    f"({results.loc[0, 'RMSE']:.2f} USD/oz).  "
    f"Tại epoch tốt nhất, validation/train loss = {gap:.2f}. "
    "Nếu tỷ số này lớn hoặc hai đường loss tách xa, cần xem lại overfitting trên đồ thị."
))

---
## 19. Dự đoán phiên kế tiếp

Dùng 60 phiên cuối của toàn bộ dữ liệu và trọng số tốt nhất đã chọn. Kết quả là minh họa kỹ thuật dự báo **phiên giao dịch kế tiếp**, không phải dự báo một ngày lịch cố định hay lời khuyên đầu tư.

In [ ]:
last_window = torch.from_numpy(features_scaled[-SEQUENCE_LENGTH:][None, :, :]).to(device)
best_model.eval()
with torch.no_grad():
    next_scaled = best_model(last_window).cpu().numpy()
next_close = float(target_scaler.inverse_transform(next_scaled)[0, 0])
latest_close = float(df["Close"].iloc[-1])
expected_change = (next_close / latest_close - 1) * 100
display(pd.DataFrame([{
    "Close gần nhất (USD/oz)": latest_close,
    "Close dự đoán phiên tiếp (USD/oz)": next_close,
    "Thay đổi dự kiến (%)": expected_change,
}]).round(2))

---
## 20. Lưu mô hình và tổng kết

Lưu trọng số, cấu hình và hai scaler để tái sử dụng đúng tiền xử lý. Thông tin tổng kết được lấy từ dữ liệu và metric vừa chạy, tránh ghi cứng một mô hình thắng cuộc.

In [ ]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
model_path = ARTIFACT_DIR / "gold_best_model.pth"
scaler_path = ARTIFACT_DIR / "gold_scaler.pkl"
torch.save({
    "model_name": best_name,
    "state_dict": best_model.cpu().state_dict(),
    "features": FEATURES,
    "sequence_length": SEQUENCE_LENGTH,
    "hidden_size": HIDDEN_SIZE,
    "num_layers": NUM_LAYERS,
    "dropout": DROPOUT,
}, model_path)
joblib.dump({"feature_scaler": feature_scaler, "target_scaler": target_scaler}, scaler_path)
display(Markdown(
    f"**Tổng kết.** Dataset: `{csv_path.name}` ({len(df):,} phiên, "
    f"{df['Date'].min():%d/%m/%Y}–{df['Date'].max():%d/%m/%Y}). "
    f"Cửa sổ: {SEQUENCE_LENGTH} phiên; feature: {', '.join(FEATURES)}; "
    "mô hình: SimpleRNN, LSTM, GRU; metric: MAE, MSE, RMSE, R², MAPE. "
    f"Mô hình RMSE thấp nhất: **{best_name}** ({results.loc[0, 'RMSE']:.2f} USD/oz). "
    "Dự báo một bước từ OHLC lịch sử có giới hạn khi thị trường biến động đột ngột. "
    f"Đã lưu `{model_path}` và `{scaler_path}`."
))